# 02 Factor Exploratory Data Analysis

## Goal
Validate that Week 3 factors cover the fixed 30-symbol universe, retain expected warm-up missing values, and have interpretable distributions. This notebook does not test strategy profitability.

## Inputs
- Processed OHLCV Parquet generated by `data_pipeline.jobs.rebuild_universe`.
- Six stored factor Parquet files under `data/factors/factor_name=*/values.parquet`.

## Guardrails
- Values at date t use data available through t only.
- Factor values are not trading signals.
- Current ticker universe and yfinance data have survivorship/data-source limitations.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from ml.factors.registry import build_default_registry
from ml.factors.storage import load_factor_values

FACTOR_NAMES = [item["name"] for item in build_default_registry().list_metadata()]
frames = [load_factor_values(name) for name in FACTOR_NAMES]
factors = pd.concat(frames, ignore_index=True)

assert set(factors.columns) == {
    "date", "symbol", "factor_name", "factor_value", "factor_version", "computed_at"
}
assert factors["symbol"].nunique() == 30
assert factors.duplicated(["date", "symbol", "factor_name", "factor_version"]).sum() == 0

factors.head()

In [ ]:
coverage = (
    factors.assign(is_missing=factors["factor_value"].isna())
    .groupby("factor_name")
    .agg(
        symbols=("symbol", "nunique"),
        rows=("factor_value", "size"),
        missing_values=("is_missing", "sum"),
        missing_rate=("is_missing", "mean"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_index()
)
coverage

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for axis, name in zip(axes.flat, FACTOR_NAMES):
    sns.histplot(
        data=factors.loc[factors["factor_name"] == name].dropna(subset=["factor_value"]),
        x="factor_value",
        bins=40,
        ax=axis,
    )
    axis.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
pivot = (
    factors.dropna(subset=["factor_value"])
    .pivot_table(index=["date", "symbol"], columns="factor_name", values="factor_value")
)
correlation = pivot.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Factor correlation on aligned non-null observations")
plt.show()

In [ ]:
symbol = "SPY"
plot_frame = factors[(factors["symbol"] == symbol) & (factors["factor_name"] == "momentum_20d")]
plot_frame.plot(x="date", y="factor_value", title=f"momentum_20d — {symbol}", figsize=(12, 4))
plt.axhline(0, color="black", linewidth=1)
plt.show()